# Session tagging (Day 1 - Day 14)

In [ ]:
import pandas as pd
import numpy as np

file_path = '../../../data/study2/behavior/GameResults_cleaned.xlsx'
df = pd.read_excel(file_path)

print(f"데이터 로드 완료: {len(df):,}개 레코드")
print(f"참가자 수: {df['participantName'].nunique()}명")
print(f"\n컬럼: {list(df.columns)}")

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

df['date'] = df['timestamp'].dt.date

df = df.sort_values(['participantName', 'timestamp'])

print("데이터 정렬 완료")
print(f"총 {len(df):,}개 레코드")

In [ ]:
# Take-home records only (exclude visit 1 / visit 2)
takehome_df = df[~df['participantName'].str.contains('_exp', na=False)].copy()

print(f"Take-home 참가자: {takehome_df['participantName'].nunique()}명")
print(f"Take-home 레코드: {len(takehome_df):,}개")

In [ ]:
# Number of days per participant
print("=" * 80)
print("Take-home 참가자별 게임 진행 날짜 수")
print("=" * 80)

date_counts = []
for participant in sorted(takehome_df['participantName'].unique()):
    participant_data = takehome_df[takehome_df['participantName'] == participant]
    unique_dates = sorted(participant_data['date'].unique())
    
    date_counts.append({
        '참가자': participant,
        '날짜 수': len(unique_dates),
        '총 레코드': len(participant_data),
        '시작일': unique_dates[0] if unique_dates else None,
        '종료일': unique_dates[-1] if unique_dates else None
    })

date_counts_df = pd.DataFrame(date_counts)
print(date_counts_df.to_string(index=False))

print(f"\n평균 날짜 수: {date_counts_df['날짜 수'].mean():.1f}일")
print(f"최소: {date_counts_df['날짜 수'].min()}일")
print(f"최대: {date_counts_df['날짜 수'].max()}일")

In [ ]:
def add_day_tags(df):
    """Tag each participant's dates as day1, day2, ..."""
    
    df_tagged = df.copy()
    df_tagged['day'] = None
    
    for participant in df_tagged['participantName'].unique():
        participant_mask = df_tagged['participantName'] == participant
        participant_data = df_tagged[participant_mask]
        
        unique_dates = sorted(participant_data['date'].unique())
        
        date_to_day = {}
        for i, date in enumerate(unique_dates, 1):
            date_to_day[date] = f'day{i}'
        
        df_tagged.loc[participant_mask, 'day'] = df_tagged.loc[participant_mask, 'date'].map(date_to_day)
    
    return df_tagged

takehome_tagged = add_day_tags(takehome_df)

print("Day 태깅 완료")
print(f"고유 Day 태그: {sorted(takehome_tagged['day'].unique())}")

In [ ]:
print("\n" + "=" * 80)
print("참가자별 Day 태깅 상세")
print("=" * 80)

for participant in sorted(takehome_tagged['participantName'].unique()):
    participant_data = takehome_tagged[takehome_tagged['participantName'] == participant]
    
    print(f"\n{participant}:")
    
    day_info = participant_data.groupby(['day', 'date']).size().reset_index(name='레코드 수')
    day_info = day_info.sort_values('date')
    
    for _, row in day_info.iterrows():
        print(f"  {row['day']:6s} - {row['date']} ({row['레코드 수']:3d}개)")
    
    total_days = participant_data['day'].nunique()
    print(f"  총 {total_days}일")

In [ ]:
# Tag visit 1 / visit 2 records
visit1_df = df[df['participantName'].str.contains('_exp1', na=False)].copy()
visit2_df = df[df['participantName'].str.contains('_exp2', na=False)].copy()

visit1_df['date'] = visit1_df['timestamp'].dt.date
visit1_df['day'] = 'visit1'

visit2_df['date'] = visit2_df['timestamp'].dt.date
visit2_df['day'] = 'visit2'

df_final = pd.concat([visit1_df, takehome_tagged, visit2_df], ignore_index=True)
df_final = df_final.sort_values(['participantName', 'timestamp'])

print(f"\n전체 데이터 통합 완료")
print(f"총 레코드: {len(df_final):,}개")
print(f"참가자 수: {df_final['participantName'].nunique()}명")
print(f"\nDay 태그 종류: {sorted(df_final['day'].unique())}")

In [ ]:
output_file = '../../../data/study2/behavior/GameResults_with_days.xlsx'
df_final.to_excel(output_file, index=False)

print("\n" + "=" * 80)
print("저장 완료")
print("=" * 80)
print(f"파일: {output_file}")
print(f"레코드: {len(df_final):,}개")
print(f"참가자: {df_final['participantName'].nunique()}명")
print(f"\n컬럼: {list(df_final.columns)}")
print("\n새로 추가된 컬럼:")
print("  - date: 날짜 (YYYY-MM-DD)")
print("  - day: 세션 태그 (visit1, day1~dayN, visit2)")

In [ ]:
# Summaryprint("\n" + "=" * 80)
print("최종 요약")
print("=" * 80)

summary = df_final.groupby('participantName').agg({
    'day': 'nunique',
    'participantName': 'count'
}).rename(columns={'day': 'Day 수', 'participantName': '레코드 수'})

summary = summary.sort_index()
print(summary)